|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 4:</h2>|<h1>The Scheduler<h1>|
|<h2>Section:</h2>|<h1>Chunked prefill<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: one token budget, two kinds of work<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

Write the mixed-batch scheduler.

One token budget per step. Decoding sequences contribute one token each;
prefilling ones contribute a slice of their prompt. No step is allowed to
become a long prefill that stalls everybody.

This is stage 11, and it is the last purely logical scheduler in the course.

In [ ]:
### run this cell

# 32 short conversations that arrive at once and settle into decoding,
# then at step 40 somebody pastes in 4096 tokens.
#   [arrival_step, prompt_len, output_len]
reqs = [[0, 64, 400] for _ in range(32)] + [[40, 4096, 100]]
print(f'{len(reqs)} requests; the big one arrives at step {reqs[-1][0]}')

# Exercise 1: the budget loop

Every step spends a fixed number of tokens. Decodes first, then as much of
the waiting prompts as still fits.

In [ ]:
def schedule(requests, budget, max_running=64):
  """requests: [[arrival_step, prompt_len, output_len], ...].
  Returns an array of (tokens_in_step, sequences_decoding_that_step)."""
  pending = sorted(requests)
  waiting, prefilling, decoding = [], [], []
  steps, t = [], 0

  while pending or waiting or prefilling or decoding:
    while pending and pending[0][0] <= t:
      _, p, o = pending.pop(0); waiting.append([p, o])
    while waiting and len(prefilling) + len(decoding) < max_running:
      waiting_r = waiting.pop(0); prefilling.append(waiting_r)

    used = 0
    n_decoding = len(decoding)      # how many users are waiting on this step
    # decodes first: they are one token each and they are latency-critical
    for d in list(decoding):
      if used >= budget: break
      used += 1; d[1] -= 1
      if d[1] == 0: decoding.remove(d)

    # then fill whatever budget is left with prompt chunks
    for pr in list(prefilling):
      if used >= budget: break
      take = min(pr[0], budget - used)
      pr[0] -= take; used += take
      if pr[0] == 0:
        prefilling.remove(pr); decoding.append(pr)

    steps.append((used, n_decoding)); t += 1
    if used == 0 and not pending: break
  return np.array(steps)

s = schedule(reqs, budget=512)
print(f'{len(s)} steps, mean {s[:,0].mean():.0f} tokens, max {s[:,0].max()} tokens')

# Exercise 2: put a clock on it

Step cost is not proportional to tokens: measured in `part4_chk_theStall`, a
step is nearly free up to a couple of hundred tokens and linear after that.
Interpolate the measurements rather than assuming.

In [ ]:
# measured on an RTX 4080 Laptop in part4_chk_theStall. Use your own.
COST = {32: 9.4, 64: 9.9, 128: 10.0, 256: 11.4, 512: 18.7,
        1024: 35.9, 2048: 74.2, 4096: 178.7}
keys = np.array(sorted(COST))
vals = np.array([COST[k] for k in keys])

def step_cost(n):
  return float(np.interp(max(n,1), keys, vals))

print(f"{'budget':>7} {'steps':>7} {'total ms':>10} {'p99 ITL':>9} {'worst ITL':>11}")
for budget in (64, 128, 256, 512, 1024, 4096):
  s   = schedule(reqs, budget)
  ms  = np.array([step_cost(x) for x in s[:,0]])
  # every decoding sequence waited this step out. THAT is the population.
  itl = np.repeat(ms, s[:,1].astype(int))
  print(f'{budget:>7} {len(s):>7} {ms.sum():>9.0f}ms {np.percentile(itl,99):>8.1f}ms {itl.max():>10.1f}ms')

# Exercise 3: the dial

Sweep the budget and plot throughput against tail latency.

In [ ]:
budgets = [64, 128, 256, 512, 1024, 2048, 4096]
tot, p99, worst = [], [], []
for b in budgets:
  s   = schedule(reqs, b)
  ms  = np.array([step_cost(x) for x in s[:,0]])
  itl = np.repeat(ms, s[:,1].astype(int))
  tot.append(ms.sum()); p99.append(np.percentile(itl, 99)); worst.append(itl.max())

fig, ax1 = plt.subplots(figsize=(7.5,4.6))
ax1.plot(budgets, tot, 'mo-', label='total time (throughput)')
ax1.set_xscale('log', base=2)
ax1.set(xlabel='Token budget per step', ylabel='Total ms to finish everything')
ax2 = ax1.twinx()
ax2.plot(budgets, p99,   'ro--', label='p99 ITL')
ax2.plot(budgets, worst, 'r^-',  label='worst ITL')
ax2.set_yscale('log')
ax2.set_ylabel('p99 ITL (ms)')
ax1.grid(alpha=.3)
fig.legend(loc='upper center'); plt.title('The dial has two ends')
plt.tight_layout(); plt.show()

print(f'fastest overall:  budget {budgets[int(np.argmin(tot))]}')
print(f'lowest p99 ITL:   budget {budgets[int(np.argmin(p99))]}')
print(f'lowest worst ITL: budget {budgets[int(np.argmin(worst))]}')

### One scheduler, two customers, and a lying statistic

Throughput and tail latency point in opposite directions, which is the
honest result and the reason this is a configuration value rather than a
constant. A big budget finishes the work sooner and makes every decoding
user wait through it.

Below a couple of hundred tokens the ITL curve goes flat, because a step
that size costs the same as a step with thirty-two. Shrinking the budget
past that buys latency you were not going to feel and costs throughput
you were. That floor is the roofline from Part 1, not the scheduler.

### Now look at the largest budget again

Its **p99 is one of the best on the table** and its **worst ITL is by far
the worst**. Both numbers are correct.

At budget 4096 the whole prompt goes through in two steps. Those two
steps are catastrophic and every one of the thirty-two users sits through
them. But two steps out of four hundred, with thirty-two users each, is
half a percent of all the token-waits in the run. It does not reach the
99th percentile. It is not even close.

This is not a flaw in the simulation. It is what happens on real
dashboards: a rare stall that every user notices, sitting quietly under
the p99 line, while the graph stays green and the complaints come in
anyway. A percentile is a statement about a population, and the
population here is token-waits, not users and not incidents.

Stage 16 is where you decide which numbers to export. Export the maximum
as well as the percentile, and know which question each one answers.

### Two details in the code that are not arbitrary

- **Decodes are served before prefill chunks.** A user watching text stop
  notices sooner than one who has not seen anything yet.
- **The prompt takes whatever budget is left**, rather than a fixed
  chunk. Nothing had to decide in advance whether this is a prefill step
  or a decode step, which is exactly what vLLM V1 generalised: there are
  no such steps any more, only a token budget.

    ./vc guide 11